
# CS 451/651: Data-Intensive Distributed Computing (Fall 2025)
# Assignment 1: Warmup

In this assignment, you'll make sure that you've correctly set up your local Spark environment.
You'll then complete a classic "Word Count" task on the `description` column of the `a1-brand.csv` file.

You can think of "Word Count" as the "Hello World!" of Hadoop, Spark, etc.
The task is simple: We want to count the total number of times each word occurs (in a potentially large collection of text).
Typically, we want to sort by the counts in descending order so we can examine the most frequently occurring words.

## 1. Setup

The following code snippet should "just work" to initialize Spark.
If it doesn't, consult [the software page](https://lintool.github.io/cs451-2025f/software.html) on the course homepage.

In [1]:
import findspark, os
os.environ["SPARK_HOME"] = "/Users/andy/Documents/spark-4.0.1-bin-hadoop3"
findspark.init()

Edit the path below to point to your local copy of `a1-brand.csv`. 

Examples:
- macOS/Linux: `/Users/yourname/data/a1-brand.csv`
- Windows: `C:\\Users\\yourname\\data\\a1-brand.csv`

In [2]:
# TODO: Set the path to a1-brand.csv
DATA_PATH = "/Users/andy/Documents/CS651/a1-brand.csv"

Import PySpark:

In [3]:
import sys, re
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.functions import col

Set up to measure wall time and memory. (Don't worry about the details, just run the cell)


In [4]:
from IPython.core.magic import register_cell_magic
import time, os, platform
import psutil, resource

def _rss_bytes():
    return psutil.Process(os.getpid()).memory_info().rss

def _ru_maxrss_bytes():
    # ru_maxrss: bytes on macOS; kilobytes on Linux
    ru = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    if platform.system() == "Darwin":
        return int(ru)  # bytes
    else:
        return int(ru) * 1024  # KB -> bytes

@register_cell_magic
def timemem(line, cell):
    """
    Measure wall time and memory around the execution of this cell.
    Usage:
        %%timemem
        <your code>
    """
    ip = get_ipython()
    rss_before = _rss_bytes()
    peak_before = _ru_maxrss_bytes()
    t0 = time.perf_counter()

    # Execute the cell body
    result = ip.run_cell(cell)

    t1 = time.perf_counter()
    rss_after = _rss_bytes()
    peak_after = _ru_maxrss_bytes()

    wall = t1 - t0
    rss_delta_mb = (rss_after - rss_before) / (1024*1024)
    peak_delta_mb = (peak_after - peak_before) / (1024*1024)

    print("======================================")
    print(f"Wall time: {wall:.3f} s")
    print(f"RSS Δ: {rss_delta_mb:+.2f} MB")
    print(f"Peak memory Δ: {peak_delta_mb:+.2f} MB (OS-dependent)")
    print("======================================")

    return result

Start a local Spark session (i.e., a `SparkContext`):

In [5]:
%%timemem

spark = (
    SparkSession.builder
    .appName("Assignment1")
    .master("local[*]")            # Use all local cores
    .config("spark.ui.showConsoleProgress", "true")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/09/22 22:48:16 WARN Utils: Your hostname, MacBook-Air-uzivatela-Andy.local, resolves to a loopback address: 127.0.0.1; using 10.10.90.108 instead (on interface en0)
25/09/22 22:48:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/22 22:48:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Wall time: 1.960 s
RSS Δ: +53.77 MB
Peak memory Δ: +53.77 MB (OS-dependent)


<ExecutionResult object at 1069f7ad0, execution_count=None error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 1069f7d10, raw_cell="
spark = (
    SparkSession.builder
    .appName(".." store_history=False silent=False shell_futures=True cell_id=None> result=<pyspark.sql.session.SparkSession object at 0x10b6d3b60>>

If you've gotten to here, congrats! Everything seems to have been set up and initialized properly!

## 2. Word Count with RDDs

First, let's read the `a1-brand.csv` file into an RDD.

<font color="red">**Here, you'll have to write some code!**</font>

**Hints:**

- You'll want to fetch the `SparkContext` from the `SparkSession`.
- There's a method of the `SparkContext` for reading in text files.
- This simple exercise should only take two lines. If you find yourself writing more code, you're doing something wrong...

In [7]:
%%timemem

# TODO: Write your code below, but do not remove any lines already in this cell.

sc = spark.sparkContext
lines = sc.textFile(DATA_PATH)


# By the time we get to here, "lines" should refer to an RDD with the brand file loaded.
# Let's count the lines.

lines.count()

7262

Wall time: 0.412 s
RSS Δ: +0.52 MB
Peak memory Δ: +0.52 MB (OS-dependent)


<ExecutionResult object at 10bac6a50, execution_count=None error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 10bac6350, raw_cell="
# TODO: Write your code below, but do not remove .." store_history=False silent=False shell_futures=True cell_id=None> result=7262>

Next, clean and tokenize text, and then find the 10 most common words.

<font color="red">**Here, you'll have to write some code!**</font>

**Required Steps:**

- Lowercase all text.
- Replace non-letter characters (`[^a-z]`) with spaces.
- Split on whitespace into tokens.
- Remove tokens with length < 2.

**Hints:**

- You _must_ use `flatMap` and other RDD operations in this step. If you're not, you're doing something wrong...
- At the end, you'll need to `collect` the output.


In [8]:
%%timemem

# TODO: Write your code below, but do not remove any lines already in this cell.


import csv

#code from piazza
from io import StringIO
def parse_csv(line):
    reader = csv.reader(
        StringIO(line),
        quotechar='"',
        escapechar='\\',
        doublequote=False
    )
    return next(reader)


#actual word count
word_counts = (
    lines
    .zipWithIndex()
    .filter(lambda x: x[1] > 0)    # skip header
    .map(lambda x: x[0])           # get the line string
    .map(parse_csv)                # parse line to fields safely
    .filter(lambda fields: len(fields) > 1)  # make sure description exists
    .map(lambda fields: fields[1]) # get description column
    .flatMap(lambda desc: re.sub(r'[^a-z]', ' ', desc.lower()).split())  # clean, tokenize
    .filter(lambda word: len(word) >= 2)  # remove short tokens
    .map(lambda word: (word, 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
    .collect()
)

# By the time we get to here "word_counts" already has the collected output, sorted by frequency in descending order.
# So we just print out the top-10.

for word, count in word_counts[:10]:
    print(f"{word}: {count}")

and: 13535
the: 7371
is: 6747
in: 6632
for: 5696
brand: 5407
its: 3438
to: 3394
of: 2798
with: 2622
Wall time: 0.293 s
RSS Δ: +2.16 MB
Peak memory Δ: +2.16 MB (OS-dependent)


<ExecutionResult object at 10bb63b60, execution_count=None error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 10bb63a70, raw_cell="
# TODO: Write your code below, but do not remove .." store_history=False silent=False shell_futures=True cell_id=None> result=None>

## 3. Word Count with DataFrames

### 3.1 Again, Just with DataFrames

Now, we're going to do the same thing, but with DataFrames instead of RDDs.

What's the difference, you ask? We'll cover it in lecture soon enough!

<font color="red">**Here, you'll have to write some code!**</font>

**Hints:**

- Here, you'll use the `SparkSession`.
- Loading a DataFrame is a single method call. If you find yourself writing more code, you're doing something wrong...
- When loading the CSV file, be aware of your escape character; use something like `.option("escape", ...)`.

In [9]:
%%timemem

# TODO: Write your code below, but do not remove any lines already in this cell.
df = spark.read.option("header", "true") \
               .option("quote", '"') \
               .option("escape", "\\") \
               .csv(DATA_PATH)

# By the time we get to here, the file should have already been loaded into a DataFrame.
# Here, we just inspect it.

print("Rows:", df.count())
df.printSchema()
df.select("description").show(5, truncate=80)

Rows: 7261
root
 |-- brand: string (nullable = true)
 |-- description: string (nullable = true)

+--------------------------------------------------------------------------------+
|                                                                     description|
+--------------------------------------------------------------------------------+
|a-case is a brand specializing in protective accessories for electronic devic...|
|A-Derma is a French dermatological skincare brand specializing in products fo...|
| a patented ingredient derived from oat plants cultivated under organic farmi...|
|                                                                       cleansers|
|           A-Derma emphasizes clinical efficacy and hypoallergenic formulations.|
+--------------------------------------------------------------------------------+
only showing top 5 rows
Wall time: 0.158 s
RSS Δ: +0.00 MB
Peak memory Δ: +0.00 MB (OS-dependent)


<ExecutionResult object at 10bb8fcb0, execution_count=None error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 10bb8fd90, raw_cell="
# TODO: Write your code below, but do not remove .." store_history=False silent=False shell_futures=True cell_id=None> result=None>

Next, clean and tokenize text, and then find the 10 most common (i.e., frequently occurring) words.
This attempts the same processing as word count with RDDs above, except here you're using a DataFrame.

<font color="red">**Here, you'll have to write some code!**</font>

**Required Steps:** (Exactly the same as above.)

- Lowercase all text.
- Replace non-letter characters (`[^a-z]`) with spaces.
- Split on whitespace into tokens.
- Remove tokens with length < 2.

**Hints:**

- You _must_ use `explode` and other Spark DataFrame operations in this exercise.
- This exercise shouldn't take more than (roughly) a dozen lines. If you find yourself writing more code, you're doing something wrong...

In [10]:
%%timemem

# TODO: Write your code below, but do not remove any lines already in this cell.

from pyspark.sql.functions import col, lower, regexp_replace, split, explode, length

# word_counts = (
#     df.select("description")  # select only the text column
#       .withColumn("description", lower(col("description")))  # lowercase
#       .withColumn("description", regexp_replace("description", "[^a-z]", " "))  # clean non-letters
#       .withColumn("word", explode(split("description", "\\s+")))  # tokenize and explode
#       .filter(length("word") >= 2)  # filter short words
#       .groupBy("word")
#       .count()
#       .orderBy(col("count").desc())
# )



import re
import csv
from io import StringIO
import os
# Error handling for bad CSV files
if "description" not in df.columns:
    raise ValueError("The CSV must contain a 'description' column.")
    
# Normalize & tokenize
cleaned = df.select(
    F.regexp_replace(F.lower(F.col("description")), "[^a-z]", " ").alias("desc")
).where(col("desc").isNotNull())

# Split into tokens
tokenized = cleaned.select(F.split(F.col("desc"), "\\s+").alias("tokens"))

# Explode tokens to one word per row
words = tokenized.select(F.explode(F.col("tokens")).alias("word")).where(col("word") != "")

# Remove short tokens
words = words.where(F.length("word") >= 2)

word_counts = (
    words.groupBy("word")
         .count()
         .orderBy(F.desc("count"), F.asc("word"))
)

# By the time we get to here "word_counts" is a DataFrame that already has the word counts sorted in descending order.
# So we just print out the top-10.

top10 = word_counts.limit(10)
top10.show()

+-----+-----+
| word|count|
+-----+-----+
|  and|13535|
|  the| 7371|
|   is| 6747|
|   in| 6632|
|  for| 5696|
|brand| 5407|
|  its| 3438|
|   to| 3394|
|   of| 2798|
| with| 2622|
+-----+-----+

Wall time: 0.291 s
RSS Δ: +0.02 MB
Peak memory Δ: +0.02 MB (OS-dependent)


<ExecutionResult object at 10bb49910, execution_count=None error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 10bb48950, raw_cell="
# TODO: Write your code below, but do not remove .." store_history=False silent=False shell_futures=True cell_id=None> result=None>

**Questions to reflect on**:

- What is conceptually different about how Spark executes `flatMap` and `explode`?
- What are the advantages or disadvantages of using each of them? 
- Are there cases where you may prefer one over the other?

(No need to write answers in the assignment submission. Just think about it...)

**Question to actually answer**:

Does the RDD approach and the DataFrame approach give the same answers? Explain why or why not.

<font color="red">**Write your answer to the above question here!**</font>

Both approaches give the same answer when being told to parse data the same way, as is the case in this notebook.
Notably, the header/brand column is included by default in the RDD approach so it has be removed manually.
It is also essential to specify the same escape characters in both sections.
Dataframes are more high level, optimized by default and tend to run RDD under the hood.
RDD is older, less optimized but allows for more low level control.
I assume these differences are the reson why the timing block is included. :)

### 3.1 Removing Stopwords

You've probably noticed that many of the most frequently occurring words are not providing us any indication about the content because they are words like "in", "the", "for", etc.
These are called stopwords.

Let's remove stopwords and count again!

<font color="red">**Here, you'll have to write some code!**</font>

**Hints:**

- Filter out all stopwords from the DataFrame before counting.
- Use `StopWordsRemover` from `pyspark.ml.feature`.

In [11]:
%%timemem

# TODO: Write your code below, but do not remove any lines already in this cell.

import numpy
from pyspark.ml.feature import StopWordsRemover

from pyspark.sql.functions import col, lower, regexp_replace, split, explode, length, size

# Clean and tokenize
df_cleaned = (
    df.select("description")
      .withColumn("description", lower(col("description")))
      .withColumn("description", regexp_replace("description", "[^a-z]", " "))
      .withColumn("tokens", split(col("description"), "\\s+"))
      .filter(col("tokens").isNotNull())   # filter out null tokens
      .filter(size(col("tokens")) > 0)    # filter out empty token arrays
)

# Remove stopwords
remover = StopWordsRemover(inputCol="tokens", outputCol="filtered_tokens")
df_no_stopwords = remover.transform(df_cleaned)

# Explode, filter, count, sort
word_counts_noStopWords = (
    df_no_stopwords
      .withColumn("word", explode("filtered_tokens"))
      .filter(length("word") >= 2)
      .groupBy("word")
      .count()
      .orderBy(col("count").desc())
)


# By the time we get to here "word_counts_noStopWords" is a DataFrame that already has the word counts sorted in descending order.
# So we just print out the top-10.

top10_noStopWords = word_counts_noStopWords.limit(10)
top10_noStopWords.show()

+------------+-----+
|        word|count|
+------------+-----+
|       brand| 5407|
|    products| 2553|
|       known| 2543|
|   primarily| 2114|
|      market| 2002|
|       range| 1717|
|  recognized| 1559|
|   including| 1482|
|specializing| 1407|
|       often| 1299|
+------------+-----+

Wall time: 0.623 s
RSS Δ: +9.23 MB
Peak memory Δ: +9.27 MB (OS-dependent)


<ExecutionResult object at 105edf330, execution_count=None error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 105f10730, raw_cell="
# TODO: Write your code below, but do not remove .." store_history=False silent=False shell_futures=True cell_id=None> result=None>

### 3.2 Saving Results to CSV

+ Save the results of the top-10 most frequently occurring words _with stopwords_, as a CSV file, to `top10_words.csv`.
+ Save the results of the top-10 frequently occurring words _discarding stopwords_, as a CSV file, to `top10_noStopWords.csv`.

<font color="red">**Here, you'll have to write some code!**</font>

In [12]:
%%timemem

# TODO: Write your code below, but do not remove any lines already in this cell.

# Save top-10 words with stopwords
word_counts.limit(10).write.mode("overwrite").csv("folder_top10_words.csv", header=True)

# Save top-10 words without stopwords
word_counts_noStopWords.limit(10).write.mode("overwrite").csv("folder_top10_noStopWords.csv", header=True)


Wall time: 0.689 s
RSS Δ: +0.03 MB
Peak memory Δ: +0.02 MB (OS-dependent)


<ExecutionResult object at 10c4b27b0, execution_count=None error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 10c4b2670, raw_cell="
# TODO: Write your code below, but do not remove .." store_history=False silent=False shell_futures=True cell_id=None> result=None>

## 4. Assignment Submission and Cleanup

For assignment submission details, see [the course homepage](https://lintool.github.io/cs451-2025f/assignments/assignment1.html).

Finally, clean up!

In [13]:
spark.stop()